# 09 — Trả lời RQ3: Huấn luyện mô hình Dự đoán (S1, S2, P1, P2, P3, T0, T1)

**Mục tiêu:** Huấn luyện Baselines, Linear và Gradient Boosting trên tập Train, dự đoán trên Test và lưu lại kết quả kiểm chứng.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


Chạy cell khởi tạo bên dưới: notebook có sẵn mã nguồn và cấu hình. Không cần mount Drive. Trên Colab, dùng `PUBG_COLAB_ALL_IN_ONE.ipynb` để giữ các bước trong cùng runtime; notebook riêng cần dữ liệu đầu ra của bước trước trong runtime hiện tại.


In [ ]:
# Bootstrap: bundled project code, no Google Drive authorization.
import base64
import io
import os
from pathlib import Path
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
_candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/Project_PUBG")]
_candidates += [p / "Project_PUBG" for p in list(_candidates)]
PROJECT_ROOT = next((p.resolve() for p in _candidates
                     if (p / "configs/data.yaml").is_file() and (p / "src/utils/config.py").is_file()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = (Path("/content") if IN_COLAB else Path.cwd()) / "Project_PUBG"
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    _bundle = zipfile.ZipFile(io.BytesIO(base64.b64decode('...')))
    for _entry in _bundle.infolist():
        _target = (PROJECT_ROOT / _entry.filename).resolve()
        if not _target.is_relative_to(PROJECT_ROOT.resolve()):
            raise ValueError("Invalid bundled path")
        if not _target.exists():
            _target.parent.mkdir(parents=True, exist_ok=True)
            _target.write_bytes(_bundle.read(_entry))
    _bundle.close()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if globals().get("PUBG_INSTALL_DEPENDENCIES", IN_COLAB) and not globals().get("_PUBG_PACKAGES_READY", False):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")])
    _PUBG_PACKAGES_READY = True

from src.utils.config import load_config, resolve_paths
cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
for _path in paths.values():
    _path.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
print("Storage:", paths["data_root"], "| Results:", paths["reports_root"])
print("No Drive mount or account token required. Export results before resetting the runtime.")


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import read_parquet_df
from src.features.registry import FeatureRegistry
from src.models.baselines import TrainMeanRegressor, TrainMedianRegressor
from src.models.linear import LinearModelWrapper
from src.models.tree_models import HistGradientBoostingWrapper
from src.models.training import train_and_predict_experiment
from src.evaluation.metrics import compute_hierarchical_metrics

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
registry = FeatureRegistry()

final_pq = paths["processed"] / "player_match_features.parquet"
split_pq = paths["interim"] / "split_assignments.parquet"

df = read_parquet_df(final_pq)
splits = read_parquet_df(split_pq)
df = df.merge(splits[["match_id", "split"]], on="match_id", how="left")

# Thí nghiệm P1 (với survival) vs P2 (bỏ survival trực tiếp)
p1_feats = registry.get_allowed_features("p1")
p2_feats = registry.get_allowed_features("p2")

print("Huấn luyện P2 (Linear Model - Không survival trực tiếp)...")
_, p2_preds = train_and_predict_experiment(df, p2_feats, "normalized_placement", LinearModelWrapper(model_type="exact"), "p2_linear", output_predictions_dir=paths["experiments"])

print("Huấn luyện P1 (Linear Model - Có survival trực tiếp)...")
_, p1_preds = train_and_predict_experiment(df, p1_feats, "normalized_placement", LinearModelWrapper(model_type="exact"), "p1_linear", output_predictions_dir=paths["experiments"])

p2_test = p2_preds[p2_preds["split"] == "test"]
p1_test = p1_preds[p1_preds["split"] == "test"]

print(f"P2 Test Micro MAE: {compute_hierarchical_metrics(p2_test)['micro']['mae']:.4f}")
print(f"P1 Test Micro MAE: {compute_hierarchical_metrics(p1_test)['micro']['mae']:.4f}")